# Session 04 — Diffusion Models**Framework:** PyTorch (CPU-friendly educational implementation)

## Objectives1. Understand the forward diffusion process.2. Implement a reverse-process approximation.3. Visualise noise progression across timesteps.4. Generate and save images using the simplified pipeline.

## Theory### Diffusion ModelsDiffusion models work in two phases:**Forward process (q):** Gradually add Gaussian noise to data over $T$ timesteps:$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\,x_{t-1},\; \beta_t I)$$**Reverse process (p):** Learn to denoise step-by-step:$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t),\; \sigma_t^2 I)$$```mermaidgraph LR    X0[Clean Image x₀] -->|add noise| X1[x₁]    X1 -->|add noise| X2[x₂]    X2 -->|...| XT[Pure Noise xₜ]    XT -->|denoise| XR2[x̂₂]    XR2 -->|denoise| XR1[x̂₁]    XR1 -->|denoise| XR0[Generated x̂₀]    style X0 fill:#2ecc71    style XT fill:#e74c3c    style XR0 fill:#3498db```

## Learning Outcomes- Implement a linear β noise schedule.- Apply forward diffusion to real images.- Approximate the reverse process.- Visualise the noising and denoising pipeline.- Save generated images to disk.

---## Setup

In [ ]:
import sysfrom pathlib import PathPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()if str(PROJECT_ROOT) not in sys.path:    sys.path.insert(0, str(PROJECT_ROOT))import torchimport matplotlib.pyplot as pltimport numpy as npfrom src.utils import seed_everything, setup_loggingfrom src.config import GENERATED_IMAGES_DIR, ensure_dirsfrom src.data_loader import get_fashion_mnist_loadersfrom src.diffusion import (    linear_beta_schedule, precompute_schedule,    forward_diffusion, forward_diffusion_sequence,    reverse_diffusion_approximation, generate_diffusion_samples,)from src.visualization import (    plot_noise_progression, plot_image_grid, save_generated_images)setup_logging()seed_everything(42)ensure_dirs()print('Setup complete ✓')

---## 1. Noise Schedule

In [ ]:
T = 300betas = linear_beta_schedule(T)schedule = precompute_schedule(betas)fig, axes = plt.subplots(1, 3, figsize=(15, 3))axes[0].plot(betas.numpy(), color='#e74c3c')axes[0].set_title('β schedule')axes[0].set_xlabel('Timestep')axes[1].plot(schedule['alphas_cumprod'].numpy(), color='#3498db')axes[1].set_title('ᾱ (cumulative product of α)')axes[1].set_xlabel('Timestep')axes[2].plot(schedule['sqrt_one_minus_alphas_cumprod'].numpy(), color='#2ecc71')axes[2].set_title('√(1 − ᾱ) — noise level')axes[2].set_xlabel('Timestep')for ax in axes:    ax.grid(True, alpha=0.3)fig.tight_layout()plt.show()

---## 2. Forward Diffusion

In [ ]:
# Get a sample image from Fashion-MNISTtrain_loader, _ = get_fashion_mnist_loaders(batch_size=1)sample_img, sample_label = next(iter(train_loader))x0 = sample_img.squeeze(0)  # (1, 28, 28)print(f'Original image shape: {x0.shape}')

In [ ]:
# Forward diffusion at selected timestepstimesteps_to_show = [0, 25, 50, 100, 150, 200, 250, 299]noisy_images = forward_diffusion_sequence(x0, timesteps_to_show, schedule)plot_noise_progression(    noisy_images,    titles=[f't={t}' for t in timesteps_to_show],    save_path=GENERATED_IMAGES_DIR / 'noise_progression.png')

### InterpretationAs $t$ increases, the image becomes progressively noisier. By $t = T-1$ the image is nearly indistinguishable from pure Gaussian noise. The forward process is fixed (not learned) — only the reverse process requires training.

---## 3. Reverse Diffusion Approximation

In [ ]:
# Start from the fully noised image and run reverse processnoisy_final, _ = forward_diffusion(x0, T - 1, schedule)denoised, snapshots = reverse_diffusion_approximation(noisy_final, schedule)print(f'Snapshots captured: {len(snapshots)}')print(f'Denoised image range: [{denoised.min():.3f}, {denoised.max():.3f}]')

In [ ]:
# Show reverse process snapshotsreverse_titles = [f'Step {i+1}' for i in range(len(snapshots))]plot_noise_progression(    snapshots,    titles=reverse_titles,    save_path=GENERATED_IMAGES_DIR / 'reverse_diffusion.png')

In [ ]:
# Side-by-side: Original → Noised → Denoisedfig, axes = plt.subplots(1, 3, figsize=(9, 3))axes[0].imshow(x0.squeeze().numpy(), cmap='gray')axes[0].set_title('Original')axes[1].imshow(noisy_final.squeeze().numpy(), cmap='gray')axes[1].set_title('Fully Noised')axes[2].imshow(denoised.squeeze().numpy(), cmap='gray')axes[2].set_title('Reverse Denoised')for ax in axes:    ax.axis('off')fig.suptitle('Forward → Reverse Diffusion', fontsize=13)fig.tight_layout()plt.show()

---## 4. Generate & Save Images

In [ ]:
# Generate 5 images using the simplified diffusion processgenerated = generate_diffusion_samples(n=5, image_size=28, timesteps=300)print(f'Generated shape: {generated.shape}')plot_image_grid(    torch.tensor(generated).unsqueeze(1), nrow=5,    title='Diffusion Model — Generated Samples',    save_path=GENERATED_IMAGES_DIR / 'diffusion_generated_grid.png')# Save individual imagessaved = save_generated_images(    torch.tensor(generated).unsqueeze(1),    prefix='diffusion_generated')print(f'\nSaved {len(saved)} images:')for p in saved:    print(f'  → {p}')

### Note on QualityThis is a *simplified educational* diffusion implementation. Real diffusion models (DDPM, Score-based) train a neural network to predict the noise $\epsilon_\theta(x_t, t)$ and require thousands of GPU hours. The outputs here demonstrate the **process**, not production quality.

---## Summary- Implemented a linear β schedule and precomputed α/ᾱ tensors.- Applied forward diffusion to Fashion-MNIST images.- Ran a simplified reverse-diffusion denoising loop.- Visualised noise progression and saved all generated images.- Real diffusion models (Stable Diffusion) train a U-Net to predict noise.

---## Interview Questions1. Explain the forward and reverse processes in a diffusion model.2. What is the role of the noise schedule (β)?3. How does DDPM differ from score-based generative models?4. Why are diffusion models slower at inference than GANs?5. What is classifier-free guidance?6. Compare diffusion models with GANs in terms of quality, diversity, and speed.7. How does latent diffusion (Stable Diffusion) improve efficiency?8. What is the connection between diffusion models and denoising score matching?

---## References1. Ho, J. et al. (2020). *Denoising Diffusion Probabilistic Models*. NeurIPS.2. Song, Y. & Ermon, S. (2020). *Score-Based Generative Modeling through SDEs*. ICLR.3. Rombach, R. et al. (2022). *High-Resolution Image Synthesis with Latent Diffusion Models*. CVPR.4. Nichol, A. & Dhariwal, P. (2021). *Improved Denoising Diffusion Probabilistic Models*. ICML.